# MethodThinker Colab 训练

一键运行训练脚本 - 直接复制整个notebook到Colab执行

In [ ]:
# 1. 检查GPU
!nvidia-smi
print("\n✓ GPU检查完成")

In [ ]:
# 2. 克隆项目
!git clone https://github.com/alasong/method-thinker.git
%cd method-thinker
print("\n✓ 项目克隆完成")

In [ ]:
# 3. 安装依赖
!pip install -q transformers accelerate peft datasets bitsandbytes trl pyyaml
print("\n✓ 依赖安装完成")

In [ ]:
# 4. 准备训练数据
# 使用方法论KB和AIME题目生成训练样本

!mkdir -p data/train_data

!python scripts/generate_training_data.py \
    --kb data/methodology_kb/v0/math_methods.yaml \
    --problems data/test_sets/aime_samples.yaml \
    --output data/train_data/train.json \
    --mode batch \
    --samples-per-problem 4

print("\n✓ 训练数据生成完成")

# 查看生成的数据
import json
with open('data/train_data/train.json', 'r') as f:
    samples = json.load(f)
print(f"生成样本数: {len(samples)}")

In [ ]:
# 5. 挂载Google Drive (保存模型用)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/method-thinker-models
print("\n✓ Drive挂载完成")

In [ ]:
# 6. 快速训练验证 (dry-run)
# 使用QLoRA 4-bit量化适合T4显存

!python scripts/train_sft.py \
    --base-model Qwen/Qwen2.5-Math-1.5B \
    --kb data/methodology_kb/v0/math_methods.yaml \
    --train-data data/train_data/train.json \
    --output-dir /content/drive/MyDrive/method-thinker-models/v1 \
    --mode methodology-injection \
    --use-lora \
    --lora-r 16 \
    --epochs 1 \
    --batch-size 4 \
    --max-length 2048 \
    --dry-run

print("\n✓ Dry-run验证完成")

In [ ]:
# 7. 正式训练 (约1-2小时)
# T4 GPU: batch_size=4 效果最佳

!python scripts/train_sft.py \
    --base-model Qwen/Qwen2.5-Math-1.5B \
    --kb data/methodology_kb/v0/math_methods.yaml \
    --train-data data/train_data/train.json \
    --output-dir /content/drive/MyDrive/method-thinker-models/v1 \
    --mode methodology-injection \
    --use-lora \
    --lora-r 16 \
    --epochs 3 \
    --batch-size 4 \
    --max-length 2048